# 6.10 · 因子分析 / Factor Analysis (FA)

> **课程定位 / Where this fits**
> PCA(6.8)是纯几何的方差最大化。因子分析长得像 PCA, 但它是一个**概率生成模型**: 假设观测变量由少数**潜在因子(latent factors)** 线性生成, 外加每个变量**各自的噪声**。它起源于心理测量学(用几道题的得分推断"智力""外向性"等看不见的因子), 也是后面 LDA 主题模型、VAE 等潜变量模型的思想源头。
> Factor Analysis is a probabilistic latent-variable model: observed variables are generated by a few latent factors plus per-variable noise. Born in psychometrics.

> 💡 **面试相关 / Interview-relevant**
> - "因子分析与 PCA 的区别" ★★★★★（概率模型 + 各异噪声 vs 几何方差）
> - "为什么 FA 对各变量噪声不同更稳健" ★★★★
> - "因子载荷(loadings) / 旋转(varimax) 是什么" ★★★
> - "FA 的生成模型假设" ★★★

---

## 学习目标 / Learning Objectives
1. FA 生成模型: $\mathbf{x}=\mathbf{Wz}+\boldsymbol\mu+\boldsymbol\epsilon$。
2. FA 与 PCA 的本质区别(各异噪声 / 概率 / 尺度不变)。
3. 因子载荷解读 + 用 FA 恢复潜在结构。
4. 何时选 FA 而非 PCA。

## 目录 / TOC
1. [生成模型 ⭐](#1)
2. [🧠 数据: 心理测量(合成)](#2)
3. [恢复潜在因子 + 载荷 ⭐](#3)
4. [FA vs PCA: 各异噪声 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 生成模型 ⭐ / The Generative Model

FA 假设每个 $d$ 维观测 $\mathbf{x}$ 由 $k$ 个($k\ll d$)潜在因子 $\mathbf{z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})$ 线性生成:
$$\mathbf{x} = \mathbf{W}\mathbf{z} + \boldsymbol\mu + \boldsymbol\epsilon, \qquad \boldsymbol\epsilon\sim\mathcal{N}(\mathbf{0},\boldsymbol\Psi)$$
- $\mathbf{W}$($d\times k$): **因子载荷矩阵** —— 每个观测变量在各因子上的权重。
- $\boldsymbol\Psi$: **对角**噪声协方差 —— 每个观测变量有**自己**的特有方差(uniqueness)。

于是 $\mathbf{x}$ 的协方差 $\text{Cov}(\mathbf{x})=\mathbf{W}\mathbf{W}^\top+\boldsymbol\Psi$。用 EM(6.6)估 $\mathbf{W},\boldsymbol\Psi$ 的 MLE。

**与 PCA 的根本区别**: PCA 把所有方向方差同等看待(隐含各向同性/等噪声), FA **显式建模每个变量不同的噪声** $\boldsymbol\Psi$ —— 所以 FA 关注的是变量间的**共享相关性(共性)**, 把各变量私有的噪声分离出去。


<a id="2"></a>
## 2. 数据: 心理测量(合成) / Synthetic Psychometric Data

模拟一份问卷: 有 2 个隐藏特质(因子)——"**学术能力**"和"**外向性**"。9 道题, 每道题主要受某一个特质驱动, 再加上**每题不同强度**的测量噪声(有的题很准, 有的很糙)。我们假装看不到特质, 只有题目得分, 用 FA 把潜在特质恢复出来。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=2, suppress=True)

rng = np.random.default_rng(0)
n = 800
academic = rng.normal(0, 1, n)      # 潜在因子1: 学术能力
extrovert = rng.normal(0, 1, n)     # 潜在因子2: 外向性
# 9 题: 前4题载学术, 后4题载外向, 第5题混合; 每题不同噪声(uniqueness)
loadings = np.array([
    [0.9,0.0],[0.8,0.0],[0.85,0.0],[0.7,0.0],   # 学术类题
    [0.5,0.5],                                   # 混合题
    [0.0,0.9],[0.0,0.8],[0.0,0.75],[0.0,0.85]])  # 外向类题
noise_sd = np.array([0.3,0.4,0.3,0.9, 0.5, 0.3,0.4,1.0,0.3])  # 每题噪声不同!
Z = np.c_[academic, extrovert]
X = Z @ loadings.T + rng.normal(0, 1, (n,9))*noise_sd
cols = [f"Q{i+1}" for i in range(9)]
df = pd.DataFrame(X, columns=cols)
print(f"问卷得分: {df.shape} (9 题, 背后 2 个潜在特质)")
print("题目相关矩阵(可见 Q1-4 / Q6-9 各自成块):")
print(df.corr().round(2).to_string())


<a id="3"></a>
## 3. 恢复潜在因子 + 载荷 ⭐ / Recovering Factors & Loadings


In [ ]:
from sklearn.decomposition import FactorAnalysis
fa = FactorAnalysis(n_components=2, random_state=0).fit(X)
est_load = fa.components_.T   # (9 题 × 2 因子) 估计载荷

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(est_load, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            yticklabels=cols, xticklabels=["因子1","因子2"], ax=ax)
ax.set_title("估计因子载荷: FA 自动发现 Q1-4 载一个因子, Q6-9 载另一个")
plt.tight_layout(); plt.show()
print("FA 恢复出两组题各自对应一个潜在因子(与我们造数据的结构一致, 符号/顺序可能不同)")

# 用估计因子分与真实特质比较 / recovered factor scores vs truth
Zhat = fa.transform(X)
c1 = np.corrcoef(Zhat[:,0], academic)[0,1]
c2 = np.corrcoef(Zhat[:,1], extrovert)[0,1]
print(f"\n估计因子 vs 真实特质 相关性: |corr|≈ {abs(c1):.2f}, {abs(c2):.2f} (高→成功恢复隐藏特质)")


<a id="4"></a>
## 4. FA vs PCA: 各异噪声 ⭐ / Heteroscedastic Noise

FA 的杀手锏: 当各变量噪声**强弱不同(异方差)**时。PCA 会被高噪声变量误导(它方差大就以为重要), FA 因为显式建模 $\boldsymbol\Psi$, 能把高噪声变量的私有噪声剔除, 更准地恢复共享结构。下面对比两者恢复潜在因子的能力。


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(X)
Zpca = pca.transform(X)

# 比较: 各方法的 2 个分量与 2 个真实因子的"最佳匹配相关性"
def best_recovery(Zest):
    cors = np.abs(np.corrcoef(np.c_[Zest, Z].T)[:2, 2:])  # 2×2
    # 每个真实因子取与某个估计分量的最大相关
    return cors.max(0).mean()

print(f"FA  恢复潜在因子的平均相关性:  {best_recovery(Zhat):.3f}")
print(f"PCA 恢复潜在因子的平均相关性:  {best_recovery(Zpca):.3f}")
print("\nFA 明显更准: 它显式建模每题不同噪声(Q4/Q8 噪声大), 把私有噪声剔除;")
print("PCA 把高噪声变量的方差当成信号, 在这种异方差噪声下被带偏。")
print("\n选择: 想要'解释/潜在结构/概率模型'用 FA; 想要'压缩/去相关/最大方差'用 PCA。")


<a id="5"></a>
## 5. 小结 / Summary

```
因子分析(FA): 概率生成模型 x = Wz + μ + ε, z~N(0,I), ε~N(0,Ψ) Ψ对角
  W=因子载荷(变量在因子上的权重); Ψ=每个变量各自的噪声(uniqueness)
  Cov(x)=WWᵀ+Ψ; 用 EM 估 MLE
vs PCA: FA 显式建模各变量不同噪声(异方差)→ 关注共享相关性, 剔除私有噪声
  PCA 几何最大方差(隐含等噪声), 高噪声变量会误导它
起源心理测量(从题目得分推断潜在特质); 可旋转(varimax)增强可解释性
```

### 💡 面试速查
1. **FA 是概率潜变量模型** x=Wz+μ+ε; PCA 是几何方差最大化
2. **FA 建模每变量不同噪声 Ψ** → 异方差噪声下比 PCA 更稳健
3. **因子载荷 W** = 变量↔因子的关系, 可旋转增可解释性
4. **选择**: 潜在结构/解释/概率→FA; 压缩/去相关/可视化→PCA
5. FA 与后续 **VAE/概率 PCA/主题模型** 同属潜变量建模思想

### 下一节
**6.11 ICA**——FA/PCA 找不相关的成分, ICA 更进一步找**统计独立**的成分。经典应用是"鸡尾酒会问题": 从混合录音里分离出各自独立的声源。
